In [ ]:
NAT_MULTS = {
    "Adamant": [1.1, 1., 0.9, 1., 1.],
    "Bashful": [1., 1., 1., 1., 1.],
    "Bold":    [0.9, 1.1, 1., 1., 1.],
    "Brave":   [1.1, 1., 1., 1., 0.9],
    "Calm":    [0.9, 1., 1., 1.1, 1.],
    "Careful": [1., 1., 0.9, 1.1, 1.],
    "Docile":  [1., 1., 1., 1., 1.],
    "Gentle":  [1., 0.9, 1., 1.1, 1.],
    "Hardy":   [1., 1., 1., 1., 1.],
    "Hasty":   [1., 0.9, 1., 1., 1.1],
    "Impish":  [1., 1.1, 0.9, 1., 1.],
    "Jolly":   [1., 1., 0.9, 1., 1.1],
    "Lax":     [1., 1.1, 1., 0.9, 1.],
    "Lonely":  [1.1, 0.9, 1., 1., 1.],
    "Mild":    [1., 0.9, 1.1, 1., 1.],
    "Modest":  [0.9, 1., 1.1, 1., 1.],
    "Naive":   [1., 1., 1., 0.9, 1.1],
    "Naughty": [1.1, 1., 1., 0.9, 1.],
    "Quiet":   [1., 1., 1.1, 1., 0.9],
    "Quirky":  [1., 1., 1., 1., 1.],
    "Rash":    [1., 1., 1.1, 0.9, 1.],
    "Relaxed": [1., 1.1, 1., 1., 0.9],
    "Sassy":   [1., 1., 1., 1.1, 0.9],
    "Serious": [1., 1., 1., 1., 1.],
    "Timid":   [0.9, 1., 1., 1., 1.1],
}

In [12]:
import copy, json, re, time
import itertools, requests
import logging
import typing

import numpy as np

from pathlib import Path
from urllib.parse import urlencode, urljoin

# First, get 'teams_full' array: [<dict1>, <dict2>], with <dict#> = { <pokename> : <dict_of_info> }
# ===========================
REPLAY_DIR = Path('./../data/test_data_replays/')

with open('./../data/POKEDEX.json', 'r') as file:
    POKEDEX = json.load(file)
with open('./../data/POKEDEX_for_test.json', 'r') as file:
    POKEDEX_for_test = json.load(file)
with open('./pokedex_raw.json', 'r') as file:
    POKEDEX_raw = json.load(file)

from battle import *

def pull_by_num(num, ns='', parse=False):
    return Battle(f'./../data/test_data_replays/gen9randombattle-{num}.json', parse)

In [11]:
POKEDEX = { item['id'] : {key:item.get(key) for key in item.keys()} for item in POKEDEX_raw }
with open('./../data/POKEDEX_for_test.json', 'w') as file:
    json.dump(POKEDEX, file)

In [3]:
d = REPLAY_DIR.iterdir()
L = [ next(d) for _ in range(500)]
for file in L : 
    if file.is_file() and file.name.endswith('.json') : 
        bat = Battle(file)
    elif file.name.startswith('.') :
        continue
    else: 
        print(f"made it to {file.name}")
        break

In [19]:
errs = []

for replay in REPLAY_DIR.iterdir(): 
    if replay.is_file() and replay.name.endswith('.json'): # skipping hidden files
        try:
            # bat = Battle(file, parse=True)
            with replay.open() as file:
                battle_json = json.load(file)

            battle_json['player_dets'] = get_player_dets(battle_json)
            TEAMS = get_teams_full(battle_json)
            
            try :
                for team in TEAMS:
                    append_team_stats(team, battle_json['id'])
            except : 
                errs.append(replay.name)
                continue
            battle_json['teams_full'] = TEAMS
            
            replay.write_text(json.dumps(battle_json), encoding='utf-8')
        
        except:
            # print("Error parsing file: %s" % replay.name)
            errs.append(replay.name)
            continue
    else:
        # print("Could not open file: %s" % replay.name)
        errs.append(replay.name)
        continue

In [21]:
# ===============================================
# Macro functions
# ===============================================
def get_teams_full(battle_json):
    '''
    Intake a replay json, compute its full teams and stats.
    '''
    
    # (1) Add 'player_dets' entry
    battle_json['player_dets'] = get_player_dets(battle_json) # [[1]]

    # (2)-(3) Get `seed`s and pass these to "server" to receive full team list.
    _temp_array = []
    
    for player in battle_json['player_dets']:
        seed = player['seed'] 
        _temp_array.append(team_from_seed(seed)) # [[2]]

    # Now we convert _temp_array -> teams_full: 
    # _temp_array[i] = [{poke_i1}, ..., {poke_i6}]
    #     |-> teams_full[i] = ['poke_i1':{poke_i1}, ..., 'poke_i6':{poke_i6}]
    teams_full = []
    for team in _temp_array:
        team_D = dict()
        for poke in team:
            team_D[poke['name']] = copy.deepcopy(poke)
        teams_full.append(team_D)

    return teams_full

# helper functions

# [[1]]
# -----------------------------------------------
def get_player_dets(battle_json):
    try:
        match1 = re.search(r'\>player p1 ({.*?})$', battle_json['inputlog'], re.M)
        p1_json = json.loads(match1.group(1))
        assert p1_json['name'] == battle_json['players'][0] # just to be certain
    except:
        print(f"Couldn't get p1 info. ({battle_json.id})")
        return None

    try:
        match2 = re.search(r'\>player p2 ({.*?})$', battle_json['inputlog'], re.M)
        p2_json = json.loads(match2.group(1))
        assert p2_json['name'] == battle_json['players'][1] # just to be certain
    except:
        print(f"Couldn't get p2 info. ({battle_json.id})")
        return None
    
    return [p1_json, p2_json]

# [[2]]
# -----------------------------------------------
def team_from_seed(seed):
    params = urlencode({
            "format": 'gen9randombattle', 
            "seed": seed
        }) 
    url = f"http://localhost:3000?{params}"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
    except requests.RequestsException as e: 
        print("Could not get team: %s", e)
        return []
    
    try: 
        team = json.loads(response.content.decode())
    except json.JSONDecodeError as e:
        logger.error("failed to parse team: %s", e)
        return []
        
    return team



# ===============================================
# add the BVs and computed Stats to each Pokemon 
def append_team_stats(team_D: dict, battle_id = ''):
    '''
    Input poke dict, compute total stats for each Pokemon in it, append to Pokemon, and return team.
    '''
    for name in team_D.keys():
        poke = team_D[name] # get the `dict`

        # (4)-(5)
        species = poke['speciesId']
        try: 
            poke['bvs'] = copy.deepcopy(POKEDEX_for_test[species]['baseStats']) # deepcopy for safety
            poke['stats'] = compute_stats(poke) # [[3]]
            poke['types'] = copy.deepcopy(POKEDEX_for_test[species].get('types'))
        except: 
            print("error with pokemon %s (%s)" % (species,battle_id))
            continue
        
    return None

# -----------------------------------------------
# [[3]]
def compute_stats(poke: dict) :
    '''
    `poke` should have dictionaries `BV`, `EV`, `IV`, and entry `level`
    example output: {'hp': 263, 'atk': 120, 'def': 169, 'spa': 240, 'spd': 216, 'spe': 223}
    '''
    _stat_D = {}

    BV = poke['bvs']
    EV = poke['evs']
    IV = poke['ivs']
    lvl = poke['level']

    for k in BV.keys():
        Q_k = (2*BV[k] + IV[k] + np.floor(EV[k]/4))*(lvl/100)
        nat_k = 1.0 # in case we want to incorporate `nature`s later
        
        if k == 'hp':
            _stat_D[k] = int(np.floor(Q_k) + lvl + 10)
        else:
            _stat_D[k] = int(np.floor((Q_k+5)*nat_k))

    return _stat_D

In [27]:
customs = []
errors = []

d = REPLAY_DIR.iterdir()
for file in d : 
    if file.is_file() and file.name.endswith('.json') : 
        try: 
            bat = Battle(file, parse=True)
        except:
            errors.append(bat.id)
        if bat.custom_ruleQ : customs.append(bat.id)
    elif file.name.startswith('.') :
        continue
    else: 
        errors.append(file.name)

0
0


In [25]:
import os

In [26]:
for cust in customs :
    os.remove('../data/test_data_replays/'+cust+'.json')